<a href="https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane as an ML task: Structured Content Archetype Clustering — this is a clustering task.

It's clustering, not classification, ranking, or scoring, because I don't have a predefined label or outcome to predict. There's no "correct answer" column in the data telling me which archetype a page belongs to — "champion," "stale visible page," "hidden gem," etc. are not columns that exist; they're groupings I have to discover from the structure of the data itself.

Not classification — classification needs known labels to train against (e.g. "is this page declining: yes/no"). I have no such labels here; archetypes aren't pre-assigned anywhere in the dataset.
Not ranking — ranking orders items against each other for a single objective (e.g. "review this page first"). I'm not producing a priority order; I'm grouping pages by shared behavior patterns.
Not scoring — scoring assigns a continuous number to each row (e.g. an opportunity score 0–100). I'm not calculating a single score per page; I'm assigning each page to one of several discovered groups.
Is clustering — I'm taking a set of observed signals (impressions, CTR, position, freshness, engagement, word count, etc.) per content item, and grouping pages that behave similarly on those signals, without any predefined answer key. The groups (archetypes) emerge from the data itself via an algorithm like K-Means, then I interpret and name what I find.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "pages loaded")
df.head()

30000 pages loaded


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target or proxy

Structured Content Archetype Clustering has no predicted target in the usual sense — there is no single outcome column I'm trying to forecast, and no historical label I'm training against.

Instead, my "proxy" is cluster membership — a group assignment that the clustering algorithm (K-Means) creates, not one that already exists in the data. Each page ends up assigned to one of k clusters based on how similar its observed signals are to other pages, where k is a number I choose.

This proxy comes from neither an observed outcome nor a predefined business rule — it's a third category: a data-driven grouping. Concretely:

It's not an observed outcome (like "did this page's traffic later decline") — I'm not measuring anything that happened after a decision point.
It's not a defined rule (like "trend_direction == down") — I'm not hand-writing a threshold or logic gate.
It's an emergent pattern — the algorithm looks at the feature space (impressions, CTR, position, freshness, word count, engagement, etc.) and groups pages that sit close together in that space. The archetype labels themselves ("champion," "stale visible page," "hidden gem," etc.) are names I apply afterward, once I inspect what's actually inside each cluster — they're descriptive, not predictive.

So more precisely: the "proxy" isn't a label I predict — it's a structure I uncover, and the naming/interpretation step is where human judgment comes in, exactly as the lane guide warns ("do not name clusters before actually inspecting what's in them").

In [ ]:
candidate_features = [
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
    "ctr", "word_count", "content_age_days", "engagement_rate", "scroll_rate"
]
df[candidate_features].describe()

,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,word_count,content_age_days,engagement_rate,scroll_rate
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,22301.000000,30000.00000,30000.000000,29875.000000
mean,5200.366300,16.097333,37.066633,16.34238,0.510733,3107.760325,256.16780,2.534520,18.212921
std,16838.019547,75.076958,107.069131,15.21679,3.279162,1452.382598,132.70793,8.310096,29.472768
min,1.000000,0.000000,1.000000,0.00000,0.000000,8.000000,90.00000,0.000000,0.000000
25%,81.000000,0.000000,2.000000,6.20000,0.000000,2413.000000,132.00000,0.000000,0.000000
50%,731.000000,1.000000,7.000000,10.80000,0.070000,2877.000000,236.00000,0.000000,5.000000
75%,3615.250000,7.000000,27.000000,22.30000,0.290000,3666.000000,333.00000,1.350000,23.530000
max,517715.000000,4178.000000,4345.000000,245.00000,100.000000,9546.000000,564.00000,100.000000,300.000000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric

My primary metric is the silhouette score.

The silhouette score measures, for every page, how similar it is to pages in its own cluster compared to pages in the next-nearest cluster. It ranges from -1 to 1:

Close to +1 — the page fits well in its assigned cluster and is clearly distinct from neighboring clusters.
Close to 0 — the page sits right on the boundary between two clusters (ambiguous).
Negative — the page is probably in the wrong cluster.

I'll average this across all pages to get one overall number for a given clustering (a given choice of k, the number of clusters). A score of roughly 0.3–0.5 is typically considered a reasonable structure for real-world, noisy data like this (SEO/content metrics are rarely as cleanly separable as, say, well-engineered synthetic data) — anything meaningfully above 0 tells me the clusters aren't just arbitrary noise, and I'll compare scores across different values of k to pick the one that separates pages best.

Why this is defensible, and why other metrics don't apply:

I can't use precision/recall/accuracy — those require true labels, and clustering has none.
I can't use Precision@K (the ranking lane's metric) — there's no ranked list, just group membership.
Silhouette score is defensible because it's a property of the feature space itself, not of my own naming choices — it doesn't just reward me for calling clusters something sensible; it mathematically checks whether the clusters are actually well-formed.

I'll also do a lightweight secondary check — cluster stability: re-running K-Means with a different random seed and confirming most pages get assigned the same cluster. If cluster membership flips wildly between runs, the "good" silhouette score wouldn't mean much, since it wouldn't be a reliable structure.

In [ ]:
from sklearn.metrics import silhouette_score
# Real computation comes once clustering is built (later section/week).
# This just confirms the metric is available and understood.
print("Metric: silhouette_score — range -1 to 1, higher = better-separated clusters")

Metric: silhouette_score — range -1 to 1, higher = better-separated clusters


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis, as a real dataframe

One row = one content item (one page), identified by content_id.

This is the natural grain for clustering archetypes, because I'm grouping pages by how they behave — not grouping days, not grouping clients, not grouping queries. Each page needs exactly one row with its own summarized signals (impressions, CTR, position, freshness, engagement, etc.) so the clustering algorithm can compare pages to each other directly.

Per the lane guide's data prep rules, I'll keep this consistent with the starter pipeline's filtering logic: keep rows where impressions_90d > 0 and content_age_days >= 90 (pages with no visibility or too little history aren't meaningful to cluster), and deduplicate by content_id so each page appears exactly once.

In [ ]:
# One row = one content item (page)
df_clean = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_clean = df_clean.drop_duplicates(subset="content_id")

print("Unit of analysis: one row = one content item (page)")
print("Rows:", df_clean.shape[0], "| Unique content_id:", df_clean["content_id"].nunique())

df_clean[["content_id", "client_id", "impressions_90d", "clicks_90d",
          "avg_position", "content_age_days", "trend_direction"]].head()

Unit of analysis: one row = one content item (page)
Rows: 30000 | Unique content_id: 30000


,content_id,client_id,impressions_90d,clicks_90d,avg_position,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,141,down
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,263,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule here

A fixed rule could sort pages on one dimension at a time — for example, "if impressions_90d > 500 and avg_position <= 10, call it a champion." But content archetypes aren't defined by one threshold; they emerge from the interaction of many signals at once, and no one can pre-write enough nested if-statements to capture that honestly.

Here's the actual mess a rule would hit:

The dimensions interact, not stack. A page with high impressions but low CTR looks completely different from a page with high impressions and high CTR — but both would trip the same "if impressions_90d > 500" rule. To capture that with if-statements, I'd need a separate branch for every meaningful combination of impressions × CTR × position × freshness × engagement × word count — that's not a handful of rules, it's potentially hundreds, and I'd be hand-guessing the thresholds for every one.
There's no natural cutoff. Where exactly does "high impressions" become "medium impressions"? Any threshold I pick (500? 1000?) is arbitrary and drawn from nowhere — it doesn't reflect where the data actually clusters. K-Means instead finds where pages naturally group together in the full feature space, rather than me guessing a boundary.
The archetypes aren't independent — they're a shape in multi-dimensional space. "Hidden gem" isn't really "high X and low Y" — it might be a specific combination across 6+ signals that's hard to even name until you see it. A fixed rule requires me to know the archetype before I define the rule; clustering lets the archetype emerge from the data, which is the whole point of this lane (I'm discovering structure, not confirming a rule I already believed).
A rule can't self-correct as data shifts. If I hand-write "thin_visible_page: word_count < 1200 and impressions_90d >= 250," that threshold is frozen — it doesn't adapt if the overall distribution of word counts or impressions changes over time or across clients. A clustering algorithm re-fit on new data reorganizes itself around whatever the current data actually looks like.
Rules can't be validated the same way. With a fixed rule, "is this a good grouping?" is unanswerable except by opinion. With clustering, I have an actual metric — silhouette score — that tells me whether the groups I found are mathematically well-separated, something a hand-written rule has no equivalent for.

The lane guide makes this exact point when it says not to name clusters before inspecting them, and to prefer effect sizes and evidence over hand-tuned scores like health_score. A rule assumes I already know the categories in advance; clustering is honest about the fact that I don't — I let the pages tell me what groups actually exist, then I name and validate what I find.

## Self-check

Before you submit, confirm each line honestly:

- [TRUE] Every section above is filled — markdown thinking AND the code that backs it
- [TRUE] The notebook runs top to bottom with no errors (Runtime → Run all)
- [TRUE] No client names, URLs, or private queries anywhere
- [TRUE] My claims use careful words: observed, measured, directional, decision-support
- [TRUE] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.